# `solveField` refactor — regression tests

This notebook tests the experimental solver implementation in `src/solve.jl` without adding it to `LowLevelFEM.jl` yet.

The development workflow is:

```julia
using LowLevelFEM
Base.include(LowLevelFEM, joinpath(dirname(pathof(LowLevelFEM)), "solve.jl"))
```

After editing only functions in `solve.jl`, rerun the include cell. If one of the guarded development structs is changed (`GlobalField`, `SolveRHS`, `NodalCoordinateSystem`), restart the Julia kernel.


In [1]:
using LowLevelFEM
using LinearAlgebra
using SparseArrays
using Test

solve_path = joinpath(
    dirname(pathof(LowLevelFEM)),
    "solve.jl"
)

@assert isfile(solve_path) "Copy solve.jl into the LowLevelFEM src directory first: $solve_path"

Base.include(
    LowLevelFEM,
    solve_path
)

using LowLevelFEM: solveField_new, Global
using LowLevelFEM: prepare_singlefield_system, solve_linear_system


In [2]:
relerr(a, b) = norm(a - b) / max(norm(a), eps(Float64))

function test_report(; kwargs...)
    return (; kwargs...)
end


test_report (generic function with 1 method)

## 1. Standard single-field solve

The first regression test compares the current package `solveField` with the experimental `solveField_new` on the same full-order 3D elasticity problem, including a non-homogeneous Dirichlet condition.


In [3]:
@testset "standard single-field" begin
    structured_box_mesh()

    mat = Material("body")
    U = Field(
        [mat],
        type=:VectorField,
        dim=3,
        fieldName=:u
    )

    K = ∫(ε(U) ⋅ D(:Solid, mat) ⋅ ε(U))
    f = ∫(U ⋅ [1.0, 0.0, 0.0], Γ="right")

    support = [
        BoundaryCondition(
            "left",
            ux=0.01,
            uy=0.0,
            uz=0.0
        )
    ]

    u_old = solveField(K, f; support=support)
    u_new = solveField_new(K, f; support=support)

    err = relerr(u_old.a, u_new.a)
    @test err < 1e-10

    display(test_report(relative_error=err))
end


(relative_error = 1.839382513048281e-15,)

Test Summary:         | Pass  Total  Time
standard single-field |    1      1  6.2s


Test.DefaultTestSet("standard single-field", false, true, false, "In[3]", 1.790770024885395e9, Random.Xoshiro(0xae7def16257f52e7, 0x0def1b5b908c8134, 0x57d31b5792d89b08, 0x3d1e375cb2c58e21, 0x3b3e4d6798c6c092), 1, 1.79077003103962e9, 0x01, ReentrantLock(), Any[])

## 2. Multiple right-hand sides

Each RHS is stored in one column. The two-column solve is compared with two separate solves.


In [4]:
@testset "multiple RHS" begin
    structured_box_mesh()

    mat = Material("body")
    U = Field([mat], type=:VectorField, dim=3, fieldName=:u)

    K = ∫(ε(U) ⋅ D(:Solid, mat) ⋅ ε(U))
    f1 = ∫(U ⋅ [1.0, 0.0, 0.0], Γ="right")
    f2 = 2.0 * f1

    support = [
        BoundaryCondition("left", ux=0.0, uy=0.0, uz=0.0)
    ]

    f_multi = VectorField(
        Matrix{Float64}[],
        hcat(f1.a[:, 1], f2.a[:, 1]),
        [0.0, 1.0],
        Int[],
        2,
        f1.type,
        f1.model
    )

    u_multi = solveField_new(K, f_multi; support=support)
    u1 = solveField_new(K, f1; support=support)
    u2 = solveField_new(K, f2; support=support)

    err1 = relerr(u_multi.a[:, 1], u1.a[:, 1])
    err2 = relerr(u_multi.a[:, 2], u2.a[:, 1])

    @test size(u_multi.a, 2) == 2
    @test err1 < 1e-12
    @test err2 < 1e-12

    display(test_report(rhs1_error=err1, rhs2_error=err2))
end


(rhs1_error = 0.0, rhs2_error = 0.0)

Test Summary: | Pass  Total  Time
multiple RHS  |    3      3  0.9s


Test.DefaultTestSet("multiple RHS", false, true, false, "In[4]", 1.790770032262033e9, Random.Xoshiro(0xae7def16257f52e7, 0x0def1b5b908c8134, 0x57d31b5792d89b08, 0x3d1e375cb2c58e21, 0x3b3e4d6798c6c092), 3, 1.790770033197882e9, 0x01, ReentrantLock(), Any[])

## 3. Symmetric matrices and direct solvers

The old symmetric `solveField` path is deliberately not used as a reference here. Instead, the experimental general solve is compared with explicit LU and Cholesky solves of the symmetric wrapper, and the reduced algebraic residual is checked.


In [5]:
@testset "symmetric direct solvers" begin
    structured_box_mesh()

    mat = Material("body")
    U = Field([mat], type=:VectorField, dim=3, fieldName=:u)

    K = ∫(ε(U) ⋅ D(:Solid, mat) ⋅ ε(U))
    f = ∫(U ⋅ [1.0, 0.0, 0.0], Γ="right")
    support = [BoundaryCondition("left", ux=0.0, uy=0.0, uz=0.0)]

    u_general = solveField_new(K, f; support=support)
    u_lu = solveField_new(
        Symmetric(K),
        f;
        support=support,
        solver=:lu
    )
    u_chol = solveField_new(
        Symmetric(K),
        f;
        support=support,
        solver=:cholesky
    )
    u_auto = solveField_new(
        Symmetric(K),
        f;
        support=support
    )

    err_lu = relerr(u_general.a, u_lu.a)
    err_chol = relerr(u_general.a, u_chol.a)
    err_auto = relerr(u_chol.a, u_auto.a)

    p = prepare_singlefield_system(
        Symmetric(K),
        f,
        support
    )
    x = solve_linear_system(p.A, p.B; solver=:cholesky)
    residual = norm(p.A * x - p.B) / norm(p.B)

    @test err_lu < 1e-10
    @test err_chol < 1e-10
    @test err_auto < 1e-12
    @test residual < 1e-10

    display(test_report(
        lu_error=err_lu,
        cholesky_error=err_chol,
        auto_error=err_auto,
        residual=residual
    ))
end


(lu_error = 1.904341338540716e-15, cholesky_error = 2.1062920393842315e-15, auto_error = 0.0, residual = 9.474664980766202e-15)

Test Summary:            | Pass  Total  Time
symmetric direct solvers |    4      4  3.1s


Test.DefaultTestSet("symmetric direct solvers", false, true, false, "In[5]", 1.790770033232588e9, Random.Xoshiro(0xae7def16257f52e7, 0x0def1b5b908c8134, 0x57d31b5792d89b08, 0x3d1e375cb2c58e21, 0x3b3e4d6798c6c092), 4, 1.790770036342638e9, 0x01, ReentrantLock(), Any[])

## 4. Iterative solvers

CG is tested on the symmetric positive-definite reduced system and GMRES on the general sparse system.


In [6]:
@testset "iterative solvers" begin
    structured_box_mesh()

    mat = Material("body")
    U = Field([mat], type=:VectorField, dim=3, fieldName=:u)

    K = ∫(ε(U) ⋅ D(:Solid, mat) ⋅ ε(U))
    f = ∫(U ⋅ [1.0, 0.0, 0.0], Γ="right")
    support = [BoundaryCondition("left", ux=0.0, uy=0.0, uz=0.0)]

    u_ref = solveField_new(K, f; support=support)

    u_cg = solveField_new(
        Symmetric(K),
        f;
        support=support,
        solver=:cg,
        reltol=1e-10,
        maxiter=size(K.A, 1)
    )

    u_gmres = solveField_new(
        K,
        f;
        support=support,
        solver=:gmres,
        reltol=1e-10,
        maxiter=size(K.A, 1)
    )

    err_cg = relerr(u_ref.a, u_cg.a)
    err_gmres = relerr(u_ref.a, u_gmres.a)

    @test err_cg < 1e-7
    @test err_gmres < 1e-7

    display(test_report(cg_error=err_cg, gmres_error=err_gmres))
end


(cg_error = 6.770884617307678e-12, gmres_error = 1.7862926702400342e-10)

Test Summary:     | Pass  Total  Time
iterative solvers |    2      2  0.7s


Test.DefaultTestSet("iterative solvers", false, true, false, "In[6]", 1.790770036384197e9, Random.Xoshiro(0xae7def16257f52e7, 0x0def1b5b908c8134, 0x57d31b5792d89b08, 0x3d1e375cb2c58e21, 0x3b3e4d6798c6c092), 2, 1.790770037100872e9, 0x01, ReentrantLock(), Any[])

## 5. H1/H2 reduced-order interpolation

A second-order mesh is assembled while the field is solved in the reduced-order space. This exercises the same `T,R` transformation pipeline used by the new solver.


In [7]:
@testset "H1/H2 reduced order" begin
    structured_box_mesh(order=2)

    mat = Material("body")
    U = Field(
        [mat],
        type=:VectorField,
        dim=3,
        fieldName=:u,
        reducedOrder=true
    )

    K = ∫(ε(U) ⋅ D(:Solid, mat) ⋅ ε(U))
    f = ∫(U ⋅ [1.0, 0.0, 0.0], Γ="right")
    support = [BoundaryCondition("left", ux=0.01, uy=0.0, uz=0.0)]

    u_old = solveField(K, f; support=support)
    u_new = solveField_new(K, f; support=support)
    u_sym = solveField_new(Symmetric(K), f; support=support)

    T, R = reductionMatrices(U)
    err = relerr(u_old.a, u_new.a)
    err_sym = relerr(u_new.a, u_sym.a)

    @test size(T, 1) == size(K.A, 1)
    @test size(T, 2) < size(T, 1)
    @test size(R) == reverse(size(T))
    @test err < 1e-10
    @test err_sym < 1e-10

    display(test_report(
        full_size=size(K.A),
        transformation_size=size(T),
        old_new_error=err,
        symmetric_error=err_sym
    ))
end


(full_size = (27783, 27783), transformation_size = (27783, 3993), old_new_error = 1.9373859141426502e-15, symmetric_error = 3.852336284142575e-15)

Test Summary:       | Pass  Total  Time
H1/H2 reduced order |    5      5  3.1s


Test.DefaultTestSet("H1/H2 reduced order", false, true, false, "In[7]", 1.790770037136653e9, Random.Xoshiro(0xae7def16257f52e7, 0x0def1b5b908c8134, 0x57d31b5792d89b08, 0x3d1e375cb2c58e21, 0x3b3e4d6798c6c092), 5, 1.790770040199186e9, 0x01, ReentrantLock(), Any[])

## 6. MPC — remote point

This self-contained 2D test mirrors the existing remote-point MPC example. The right-edge x-displacements are tied to one remote x-displacement. The remote y DOF is fixed because it is not coupled to the solid.


In [8]:
@testset "MPC remote point" begin
    structured_rect_mesh()

    remote_tag = gmsh.model.occ.addPoint(1.2, 0.5, 0.0)
    gmsh.model.occ.synchronize()
    gmsh.model.addPhysicalGroup(0, [remote_tag], -1, "remote")

    gmsh.model.mesh.clear()
    gmsh.model.mesh.generate(2)

    mat = Material("body")
    remote = Material("remote")

    U = Field(
        [mat, remote],
        type=:VectorField,
        dim=2,
        fieldName=:u,
        rhsName=:f
    )

    C = [
        2.0 1.0 0.0
        1.0 2.0 0.0
        0.0 0.0 1.0
    ]

    K = ∫(SymGrad(U) ⋅ C ⋅ SymGrad(U), Ω="body")
    f = ∫(U ⋅ [1.0, 0.0], Γ="right")

    mpc = MPC(
        master="remote",
        slave="right",
        field=U,
        ux=true,
        uy=false
    )

    support = [
        BoundaryCondition("left", ux=0.0, uy=0.0),
        BoundaryCondition("remote", uy=0.0)
    ]

    u_old = solveField(K, f; support=support, mpc=[mpc])
    u_new = solveField_new(K, f; support=support, mpc=[mpc])

    err = relerr(u_old.a, u_new.a)
    @test err < 1e-10

    # Non-homogeneous prescribed master displacement.
    support_nonhom = [
        BoundaryCondition("left", ux=0.0, uy=0.0),
        BoundaryCondition("remote", ux=0.01, uy=0.0)
    ]

    f0 = 0.0 * f

    u_old_nh = solveField(
        K,
        f0;
        support=support_nonhom,
        mpc=[mpc]
    )

    u_new_nh = solveField_new(
        K,
        f0;
        support=support_nonhom,
        mpc=[mpc]
    )

    err_nh = relerr(u_old_nh.a, u_new_nh.a)
    @test err_nh < 1e-10

    display(test_report(
        homogeneous_error=err,
        nonhomogeneous_error=err_nh
    ))
end


(homogeneous_error = 0.0, nonhomogeneous_error = 0.0)

Test Summary:    | Pass  Total  Time
MPC remote point |    2      2  1.3s


Test.DefaultTestSet("MPC remote point", false, true, false, "In[8]", 1.790770040233101e9, Random.Xoshiro(0xae7def16257f52e7, 0x0def1b5b908c8134, 0x57d31b5792d89b08, 0x3d1e375cb2c58e21, 0x3b3e4d6798c6c092), 2, 1.790770041504997e9, 0x01, ReentrantLock(), Any[])

## 7. VectorField-based coordinate system

The local first direction on the right face is chosen as global `+y`. Therefore a local load `[1,0,0]` must be identical to a globally specified load `[0,1,0]` wrapped in `Global(...)`.


In [9]:
@testset "coordinate system: local vs global load" begin
    structured_box_mesh()

    mat = Material("body")
    U = Field([mat], type=:VectorField, dim=3, fieldName=:u)

    K = ∫(ε(U) ⋅ D(:Solid, mat) ⋅ ε(U))
    support = [BoundaryCondition("left", ux=0.0, uy=0.0, uz=0.0)]

    e1 = VectorField(U, "right", [0.0, 1.0, 0.0])
    e2 = VectorField(U, "right", [-1.0, 0.0, 0.0])
    cs = CoordinateSystem(e1, e2)

    f_local = ∫(U ⋅ [1.0, 0.0, 0.0], Γ="right")
    f_global = ∫(U ⋅ [0.0, 1.0, 0.0], Γ="right")

    u_local = solveField_new(
        K,
        f_local;
        support=support,
        coordSys=[cs]
    )

    u_global = solveField_new(
        K,
        Global(f_global);
        support=support,
        coordSys=[cs]
    )

    err = relerr(u_local.a, u_global.a)
    @test err < 1e-12

    display(test_report(relative_error=err))
end


(relative_error = 0.0,)

Test Summary:                           | Pass  Total  Time
coordinate system: local vs global load |    1      1  1.2s


Test.DefaultTestSet("coordinate system: local vs global load", false, true, false, "In[9]", 1.79077004152524e9, Random.Xoshiro(0xae7def16257f52e7, 0x0def1b5b908c8134, 0x57d31b5792d89b08, 0x3d1e375cb2c58e21, 0x3b3e4d6798c6c092), 1, 1.790770042761364e9, 0x01, ReentrantLock(), Any[])

## 8. Mixed local and global loads

This checks the symbolic RHS expression `f_local + Global(f_global)`. With homogeneous essential boundary conditions, the mixed solution must equal the superposition of the two separate solutions.


In [10]:
@testset "mixed local/global RHS" begin
    structured_box_mesh()

    mat = Material("body")
    U = Field([mat], type=:VectorField, dim=3, fieldName=:u)

    K = ∫(ε(U) ⋅ D(:Solid, mat) ⋅ ε(U))
    support = [BoundaryCondition("left", ux=0.0, uy=0.0, uz=0.0)]

    e1 = VectorField(U, "right", [0.0, 1.0, 0.0])
    e2 = VectorField(U, "right", [-1.0, 0.0, 0.0])
    cs = CoordinateSystem(e1, e2)

    f_surface_local = ∫(U ⋅ [1.0, 0.0, 0.0], Γ="right")
    f_body_global = ∫(U ⋅ [0.0, 0.0, -1.0], Ω="body")

    u_surface = solveField_new(
        K,
        f_surface_local;
        support=support,
        coordSys=[cs]
    )

    u_body = solveField_new(
        K,
        Global(f_body_global);
        support=support,
        coordSys=[cs]
    )

    u_mixed = solveField_new(
        K,
        f_surface_local + Global(f_body_global);
        support=support,
        coordSys=[cs]
    )

    err = relerr(u_mixed.a, u_surface.a + u_body.a)
    @test err < 1e-10

    display(test_report(superposition_error=err))
end


(superposition_error = 9.493682772719504e-16,)

Test Summary:          | Pass  Total  Time
mixed local/global RHS |    1      1  1.3s


Test.DefaultTestSet("mixed local/global RHS", false, true, false, "In[10]", 1.790770042781975e9, Random.Xoshiro(0xae7def16257f52e7, 0x0def1b5b908c8134, 0x57d31b5792d89b08, 0x3d1e375cb2c58e21, 0x3b3e4d6798c6c092), 1, 1.790770044051449e9, 0x01, ReentrantLock(), Any[])

## 9. Inclined roller support

The right face receives a local first axis at 45° in the global x–y plane. `BoundaryCondition("right", ux=0)` therefore constrains displacement along that local axis. In global components this implies approximately `ux + uy = 0` on the right face.


In [11]:
@testset "inclined roller" begin
    structured_box_mesh()

    mat = Material("body")
    U = Field([mat], type=:VectorField, dim=3, fieldName=:u)

    K = ∫(ε(U) ⋅ D(:Solid, mat) ⋅ ε(U))

    e1 = VectorField(U, "right", [1.0, 1.0, 0.0])
    e2 = VectorField(U, "right", [-1.0, 1.0, 0.0])
    cs45 = CoordinateSystem(e1, e2)

    support = [
        BoundaryCondition("left", ux=0.0, uy=0.0, uz=0.0),
        BoundaryCondition("right", ux=0.0)
    ]

    f_global = ∫(U ⋅ [0.0, 1.0, 0.0], Γ="right")

    u = solveField_new(
        K,
        Global(f_global);
        support=support,
        coordSys=[cs45]
    )

    right_nodes = LowLevelFEM._nodesOnPhysicalGroup(U, "right")
    ux_dofs = 3 .* (right_nodes .- 1) .+ 1
    uy_dofs = 3 .* (right_nodes .- 1) .+ 2

    roller_error = maximum(abs, u.a[ux_dofs, 1] + u.a[uy_dofs, 1])
    scale = maximum(abs, u.a[:, 1])
    relative_roller_error = roller_error / max(scale, eps(Float64))

    @test relative_roller_error < 1e-10

    display(test_report(
        absolute_constraint_error=roller_error,
        relative_constraint_error=relative_roller_error
    ))
end


(absolute_constraint_error = 0.0, relative_constraint_error = 0.0)

Test Summary:   | Pass  Total  Time
inclined roller |    1      1  0.6s


Test.DefaultTestSet("inclined roller", false, true, false, "In[11]", 1.790770044077163e9, Random.Xoshiro(0xae7def16257f52e7, 0x0def1b5b908c8134, 0x57d31b5792d89b08, 0x3d1e375cb2c58e21, 0x3b3e4d6798c6c092), 1, 1.790770044644077e9, 0x01, ReentrantLock(), Any[])

## 10. `normalVector` as a coordinate-system direction

This is a smoke test for the intended high-level workflow. The surface normal is used directly as `e1`; a simple tangential VectorField supplies `e2`. The resulting nodal transformation should be orthogonal.


In [12]:
@testset "normalVector coordinate system" begin
    structured_box_mesh()

    mat = Material("body")
    U = Field([mat], type=:VectorField, dim=3, fieldName=:u)

    e1 = normalVector(U, "right")
    e2 = VectorField(U, "right", [0.0, 1.0, 0.0])
    cs = CoordinateSystem(e1, e2)

    Q = LowLevelFEM._build_coordinate_transformation(U, [cs])
    Iq = spdiagm(0 => ones(Float64, size(Q.T, 1)))

    orthogonality_error = norm(Q.T' * Q.T - Iq) / norm(Iq)
    @test orthogonality_error < 1e-12

    display(test_report(orthogonality_error=orthogonality_error))
end


(orthogonality_error = 0.0,)

Test Summary:                  | Pass  Total  Time
normalVector coordinate system |    1      1  0.2s


Test.DefaultTestSet("normalVector coordinate system", false, true, false, "In[12]", 1.790770044665413e9, Random.Xoshiro(0xae7def16257f52e7, 0x0def1b5b908c8134, 0x57d31b5792d89b08, 0x3d1e375cb2c58e21, 0x3b3e4d6798c6c092), 1, 1.790770044865587e9, 0x01, ReentrantLock(), Any[])

## Current intentional limitations

The following combinations are intentionally rejected by `solve.jl` for now:

- `coordSys` + `reducedOrder=true`
- `coordSys` + MPC
- overlapping coordinate-system supports

Those cases need an explicit decision about the physical meaning and ordering of the coordinate-basis transformation relative to the kinematic reduction.
